## Quantum Adder circuit

In [2]:
from gas.gas_circuit import adder_circuit, bin_to_int_twos_compliment
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import QFT

/Users/tom/Documents/ucl/projects/GAS/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
n_qubits = 10

low  = -2**(n_qubits-1)
high =  2**(n_qubits-1)
print(f"allowed range on register: [{low}, {high}) -> with these being the max values that can be stored")

allowed_range = np.arange(low, high)

allowed range on register: [-512, 512) -> with these being the max values that can be stored


In [4]:
XY_plane_circuit = QuantumCircuit(n_qubits)
for qi in range(XY_plane_circuit.num_qubits):
    XY_plane_circuit.h(qi)
XY_plane_circuit.draw()

┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     ├───┤
q_2: ┤ H ├
     ├───┤
q_3: ┤ H ├
     ├───┤
q_4: ┤ H ├
     ├───┤
q_5: ┤ H ├
     ├───┤
q_6: ┤ H ├
     ├───┤
q_7: ┤ H ├
     ├───┤
q_8: ┤ H ├
     ├───┤
q_9: ┤ H ├
     └───┘

In [5]:
# Can change these values
value_1 = 12 
value_2 = -28

# assert statements makes sure values do not overflow register
assert value_1 in allowed_range, "value_1 not in allowed range"
assert value_2 in allowed_range, "value_2 not in allowed range"
assert (value_1+value_2) in allowed_range, "the sum of these values not in allowed range"

In [6]:
adder_1 = adder_circuit(n_qubits, value_1, label=f"{value_1}")
full_qc = XY_plane_circuit.compose(adder_1)
full_qc.draw()

┌───┐┌─────┐
q_0: ┤ H ├┤0    ├
     ├───┤│     │
q_1: ┤ H ├┤1    ├
     ├───┤│     │
q_2: ┤ H ├┤2    ├
     ├───┤│     │
q_3: ┤ H ├┤3    ├
     ├───┤│     │
q_4: ┤ H ├┤4    ├
     ├───┤│  12 │
q_5: ┤ H ├┤5    ├
     ├───┤│     │
q_6: ┤ H ├┤6    ├
     ├───┤│     │
q_7: ┤ H ├┤7    ├
     ├───┤│     │
q_8: ┤ H ├┤8    ├
     ├───┤│     │
q_9: ┤ H ├┤9    ├
     └───┘└─────┘

In [7]:
# # can use this code to see the phase gates
# from qiskit import transpile
# under_the_hood = transpile(full_qc, basis_gates=["h", "p"])
# under_the_hood.draw()

In [8]:
QFT_dag = QFT(num_qubits=n_qubits, inverse=True)
qc = full_qc.compose(QFT_dag)
qc.measure_all()
qc.draw()

┌───┐┌─────┐┌───────┐ ░ ┌─┐                           
    q_0: ┤ H ├┤0    ├┤0      ├─░─┤M├───────────────────────────
         ├───┤│     ││       │ ░ └╥┘┌─┐                        
    q_1: ┤ H ├┤1    ├┤1      ├─░──╫─┤M├────────────────────────
         ├───┤│     ││       │ ░  ║ └╥┘┌─┐                     
    q_2: ┤ H ├┤2    ├┤2      ├─░──╫──╫─┤M├─────────────────────
         ├───┤│     ││       │ ░  ║  ║ └╥┘┌─┐                  
    q_3: ┤ H ├┤3    ├┤3      ├─░──╫──╫──╫─┤M├──────────────────
         ├───┤│     ││       │ ░  ║  ║  ║ └╥┘┌─┐               
    q_4: ┤ H ├┤4    ├┤4      ├─░──╫──╫──╫──╫─┤M├───────────────
         ├───┤│  12 ││  IQFT │ ░  ║  ║  ║  ║ └╥┘┌─┐            
    q_5: ┤ H ├┤5    ├┤5      ├─░──╫──╫──╫──╫──╫─┤M├────────────
         ├───┤│     ││       │ ░  ║  ║  ║  ║  ║ └╥┘┌─┐         
    q_6: ┤ H ├┤6    ├┤6      ├─░──╫──╫──╫──╫──╫──╫─┤M├─────────
         ├───┤│     ││       │ ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐      
    q_7: ┤ H ├┤7    ├┤7      ├─░──╫──╫──╫──╫──╫──╫──╫─┤M├──────
         ├───┤│     ││       │ ░  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐   
    q_8: ┤ H ├┤8    ├┤8      ├─░──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───
         ├───┤│     ││       │ ░  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐
    q_9: ┤ H ├┤9    ├┤9      ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├
         └───┘└─────┘└───────┘ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘
meas: 10/═════════════════════════╩══╩══╩══╩══╩══╩══╩══╩══╩══╩═
                                  0  1  2  3  4  5  6  7  8  9

In [9]:
print(f"{value_1} in quantum circuit:")

sampler = StatevectorSampler()
job = sampler.run([qc], shots=100)
out = job.result()[0].data.meas.get_counts()
out

12 in quantum circuit:


{'0000001100': 100}

In [10]:
outbitstring = np.array(list(list(out.keys())[0])).astype(np.uint8)
bin_to_int_twos_compliment(outbitstring)

np.int64(12)

In [11]:
## Now do some aritmetic

In [12]:
adder_2 = adder_circuit(n_qubits, value_2, label=f"{value_2}")
full_qc2 = full_qc.compose(adder_2)
full_qc2.draw()

┌───┐┌─────┐┌──────┐
q_0: ┤ H ├┤0    ├┤0     ├
     ├───┤│     ││      │
q_1: ┤ H ├┤1    ├┤1     ├
     ├───┤│     ││      │
q_2: ┤ H ├┤2    ├┤2     ├
     ├───┤│     ││      │
q_3: ┤ H ├┤3    ├┤3     ├
     ├───┤│     ││      │
q_4: ┤ H ├┤4    ├┤4     ├
     ├───┤│  12 ││  -28 │
q_5: ┤ H ├┤5    ├┤5     ├
     ├───┤│     ││      │
q_6: ┤ H ├┤6    ├┤6     ├
     ├───┤│     ││      │
q_7: ┤ H ├┤7    ├┤7     ├
     ├───┤│     ││      │
q_8: ┤ H ├┤8    ├┤8     ├
     ├───┤│     ││      │
q_9: ┤ H ├┤9    ├┤9     ├
     └───┘└─────┘└──────┘

In [13]:
QFT_dag = QFT(num_qubits=n_qubits, inverse=True)
qc2 = full_qc2.compose(QFT_dag)
qc2.measure_all()
qc2.draw()

┌───┐┌─────┐┌──────┐┌───────┐ ░ ┌─┐                           
    q_0: ┤ H ├┤0    ├┤0     ├┤0      ├─░─┤M├───────────────────────────
         ├───┤│     ││      ││       │ ░ └╥┘┌─┐                        
    q_1: ┤ H ├┤1    ├┤1     ├┤1      ├─░──╫─┤M├────────────────────────
         ├───┤│     ││      ││       │ ░  ║ └╥┘┌─┐                     
    q_2: ┤ H ├┤2    ├┤2     ├┤2      ├─░──╫──╫─┤M├─────────────────────
         ├───┤│     ││      ││       │ ░  ║  ║ └╥┘┌─┐                  
    q_3: ┤ H ├┤3    ├┤3     ├┤3      ├─░──╫──╫──╫─┤M├──────────────────
         ├───┤│     ││      ││       │ ░  ║  ║  ║ └╥┘┌─┐               
    q_4: ┤ H ├┤4    ├┤4     ├┤4      ├─░──╫──╫──╫──╫─┤M├───────────────
         ├───┤│  12 ││  -28 ││  IQFT │ ░  ║  ║  ║  ║ └╥┘┌─┐            
    q_5: ┤ H ├┤5    ├┤5     ├┤5      ├─░──╫──╫──╫──╫──╫─┤M├────────────
         ├───┤│     ││      ││       │ ░  ║  ║  ║  ║  ║ └╥┘┌─┐         
    q_6: ┤ H ├┤6    ├┤6     ├┤6      ├─░──╫──╫──╫──╫──╫──╫─┤M├─────────
         ├───┤│     ││      ││       │ ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐      
    q_7: ┤ H ├┤7    ├┤7     ├┤7      ├─░──╫──╫──╫──╫──╫──╫──╫─┤M├──────
         ├───┤│     ││      ││       │ ░  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐   
    q_8: ┤ H ├┤8    ├┤8     ├┤8      ├─░──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───
         ├───┤│     ││      ││       │ ░  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐
    q_9: ┤ H ├┤9    ├┤9     ├┤9      ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├
         └───┘└─────┘└──────┘└───────┘ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘
meas: 10/═════════════════════════════════╩══╩══╩══╩══╩══╩══╩══╩══╩══╩═
                                          0  1  2  3  4  5  6  7  8  9

In [14]:
sampler = StatevectorSampler()
job = sampler.run([qc2], shots=100)
out = job.result()[0].data.meas.get_counts()
out

{'1111110000': 100}

In [15]:
print(f"{value_1} + {value_2} in quantum circuit:")
outbitstring = np.array(list(list(out.keys())[0])).astype(np.uint8)
bin_to_int_twos_compliment(outbitstring)

12 + -28 in quantum circuit:


np.int64(-16)

In [16]:
### combine everything
## this code will randomly generate a list of ints to add in a quantum circuit!


# 1. generate random terms and ensure they do not overflow the register
values_to_add = np.random.randint(-10,10, size=20) ## generate a random list of ints (-10,10) of size 20 between 
assert np.abs(values_to_add).sum() < 2**(n_qubits-1), "adding values that overflow register"

#2. Apply Hadamard to go into XY plane
qc_list_add = QuantumCircuit(n_qubits)
for qi in range(qc_list_add.num_qubits):
    qc_list_add.h(qi)


#3. construct each adder subcircuit (using phase gates) and append to the overall circuit
for val in values_to_add:
    adder = adder_circuit(n_qubits, val, label=f"{val}")
    qc_list_add = qc_list_add.compose(adder)

#4. Map from Fourier basis to computational basis using inverse QFT
QFT_dag = QFT(num_qubits=n_qubits, inverse=True)
qc_list_add = qc_list_add.compose(QFT_dag)
qc_list_add.measure_all()
qc_list_add.draw()

┌───┐┌────┐┌────┐┌────┐┌──────┐┌─────┐┌────┐┌────┐┌──────┐┌────┐»
    q_0: ┤ H ├┤0   ├┤0   ├┤0   ├┤0     ├┤0    ├┤0   ├┤0   ├┤0     ├┤0   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_1: ┤ H ├┤1   ├┤1   ├┤1   ├┤1     ├┤1    ├┤1   ├┤1   ├┤1     ├┤1   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_2: ┤ H ├┤2   ├┤2   ├┤2   ├┤2     ├┤2    ├┤2   ├┤2   ├┤2     ├┤2   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_3: ┤ H ├┤3   ├┤3   ├┤3   ├┤3     ├┤3    ├┤3   ├┤3   ├┤3     ├┤3   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_4: ┤ H ├┤4   ├┤4   ├┤4   ├┤4     ├┤4    ├┤4   ├┤4   ├┤4     ├┤4   ├»
         ├───┤│  0 ││  9 ││  2 ││  -10 ││  -1 ││  6 ││  4 ││  -10 ││  2 │»
    q_5: ┤ H ├┤5   ├┤5   ├┤5   ├┤5     ├┤5    ├┤5   ├┤5   ├┤5     ├┤5   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_6: ┤ H ├┤6   ├┤6   ├┤6   ├┤6     ├┤6    ├┤6   ├┤6   ├┤6     ├┤6   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_7: ┤ H ├┤7   ├┤7   ├┤7   ├┤7     ├┤7    ├┤7   ├┤7   ├┤7     ├┤7   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_8: ┤ H ├┤8   ├┤8   ├┤8   ├┤8     ├┤8    ├┤8   ├┤8   ├┤8     ├┤8   ├»
         ├───┤│    ││    ││    ││      ││     ││    ││    ││      ││    │»
    q_9: ┤ H ├┤9   ├┤9   ├┤9   ├┤9     ├┤9    ├┤9   ├┤9   ├┤9     ├┤9   ├»
         └───┘└────┘└────┘└────┘└──────┘└─────┘└────┘└────┘└──────┘└────┘»
meas: 10/════════════════════════════════════════════════════════════════»
                                                                         »
«         ┌─────┐┌────┐┌────┐┌─────┐┌────┐┌─────┐┌────┐┌──────┐┌─────┐┌────┐»
«    q_0: ┤0    ├┤0   ├┤0   ├┤0    ├┤0   ├┤0    ├┤0   ├┤0     ├┤0    ├┤0   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_1: ┤1    ├┤1   ├┤1   ├┤1    ├┤1   ├┤1    ├┤1   ├┤1     ├┤1    ├┤1   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_2: ┤2    ├┤2   ├┤2   ├┤2    ├┤2   ├┤2    ├┤2   ├┤2     ├┤2    ├┤2   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_3: ┤3    ├┤3   ├┤3   ├┤3    ├┤3   ├┤3    ├┤3   ├┤3     ├┤3    ├┤3   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_4: ┤4    ├┤4   ├┤4   ├┤4    ├┤4   ├┤4    ├┤4   ├┤4     ├┤4    ├┤4   ├»
«         │  -9 ││  9 ││  7 ││  -4 ││  3 ││  -2 ││  3 ││  -10 ││  -6 ││  7 │»
«    q_5: ┤5    ├┤5   ├┤5   ├┤5    ├┤5   ├┤5    ├┤5   ├┤5     ├┤5    ├┤5   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_6: ┤6    ├┤6   ├┤6   ├┤6    ├┤6   ├┤6    ├┤6   ├┤6     ├┤6    ├┤6   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_7: ┤7    ├┤7   ├┤7   ├┤7    ├┤7   ├┤7    ├┤7   ├┤7     ├┤7    ├┤7   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_8: ┤8    ├┤8   ├┤8   ├┤8    ├┤8   ├┤8    ├┤8   ├┤8     ├┤8    ├┤8   ├»
«         │     ││    ││    ││     ││    ││     ││    ││      ││     ││    │»
«    q_9: ┤9    ├┤9   ├┤9   ├┤9    ├┤9   ├┤9    ├┤9   ├┤9     ├┤9    ├┤9   ├»
«         └─────┘└────┘└────┘└─────┘└────┘└─────┘└────┘└──────┘└─────┘└────┘»
«meas: 10/══════════════════════════════════════════════════════════════════»
«                                                                           »
«         ┌─────┐┌───────┐ ░ ┌─┐                           
«    q_0: ┤0    ├┤0      ├─░─┤M├───────────────────────────
«         │     ││       │ ░ └╥┘┌─┐                        
«    q_1: ┤1    ├┤1      ├─░──╫─┤M├────────────────────────
«         │     ││       │ ░  ║ └╥┘┌─┐                     
«    q_2: ┤2    ├┤2      ├─░──╫──╫─┤M├─────────────────────
«         │     ││       │ ░  ║  ║ └╥┘┌─┐                  
«    q_3: ┤3    ├┤3      ├─░──╫──╫──╫─┤M├──────────────────
«         

In [17]:
#5. Simulate the result
sampler = StatevectorSampler()
job_list_add = sampler.run([qc_list_add], shots=100)
out_list_add = job_list_add.result()[0].data.meas.get_counts()

outbitstring = np.array(list(list(out_list_add.keys())[0])).astype(np.uint8)

print(f"QC output: {bin_to_int_twos_compliment(outbitstring)}")
print(f"list sum : {np.sum(values_to_add)}")

QC output: -9
list sum : -9
